# GrowWithMe — Offline Nana NLU (Tier 1.5)

Distills the online Nana LLM's *understanding* into a ~1 MB on-device model so that,
offline, the caregiver app still understands natural Ghanaian-English requests —
"she has been shaking since morning and won't take breast" → `start_health_check`
(subject: child) — instead of today's keyword matching. Replies stay curated/vetted
in the app; this model NEVER generates text, so it cannot hallucinate.

**Run top to bottom on Colab (CPU fine, ~10 min).** The NVIDIA distillation cell is
optional — with no API key the model trains on the built-in templates alone (still good;
the LLM data adds phrasing diversity).

Outputs: `nana_nlu.tflite` + `manifest.json`, registered as **nana-nlu** in the model registry.

## The featurizer contract (MUST match `mobile/lib/data/model/nlu_service.dart`)
1. lowercase; replace every char not in `[a-z0-9' ]` with a space; collapse spaces; trim
2. tokens = split on single space
3. features = word unigrams `u:<tok>`, word bigrams `b:<t1>_<t2>`, char trigrams `c:<3 chars>`
   of each token padded as `^tok$`
4. bucket = FNV-1a 32-bit hash of the UTF-8 feature string, modulo **8192**
5. vector = bucket counts, then L2-normalized
6. model output = single tensor: `[10 intent probs] + [3 subject probs]`

In [ ]:
%pip -q install tensorflow scikit-learn numpy requests
import hashlib, json, random, re
import numpy as np, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
random.seed(42); np.random.seed(42); tf.random.set_seed(42)

INTENTS = ['start_health_check', 'open_add_child', 'open_add_pregnancy', 'plan_diet',
           'read_today', 'get_tip', 'log_weight', 'set_reminder', 'greeting', 'help_other']
SUBJECTS = ['child', 'pregnancy', 'unknown']
BUCKETS = 8192

def fnv1a32(s):
    h = 2166136261
    for b in s.encode('utf-8'):
        h ^= b
        h = (h * 16777619) & 0xFFFFFFFF
    return h

def featurize(text):
    t = re.sub(r'[^a-z0-9\' ]', ' ', text.lower())
    toks = [w for w in t.split(' ') if w]
    feats = []
    for w in toks:
        feats.append('u:' + w)
        p = '^' + w + '$'
        feats += ['c:' + p[i:i+3] for i in range(len(p) - 2)]
    feats += ['b:' + a + '_' + b for a, b in zip(toks, toks[1:])]
    v = np.zeros(BUCKETS, dtype='float32')
    for f in feats:
        v[fnv1a32(f) % BUCKETS] += 1
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

# Contract check — the Dart test asserts these SAME numbers.
probe = featurize('my baby has fever')
print('nonzero:', int((probe > 0).sum()), 'first idx:', int(np.argmax(probe > 0)))
print('fnv1a32("u:fever") % 8192 =', fnv1a32('u:fever') % BUCKETS)

## 1. Template data — guaranteed coverage of every intent, Ghana-flavored

In [ ]:
CHILD_SYMPTOMS = ['fever', 'hot body', 'convulsions', 'fits', 'shaking', 'vomiting everything',
  'diarrhoea', 'watery stool', 'blood in stool', 'coughing', 'fast breathing', 'chest pulling in',
  'not eating', 'refusing breast', "won't take breast", 'very weak', 'sleeping too much',
  'yellow eyes', 'rash all over', 'crying nonstop', 'sunken eyes']
PREG_SYMPTOMS = ['bleeding', 'spotting blood', 'severe headache', 'blurred vision',
  'swollen feet', 'swollen face and hands', 'baby not moving', 'baby stopped kicking',
  'strong belly pain', 'water broke early', 'fever and chills', 'too weak to stand',
  'dizzy all the time', 'vomiting everything I eat']
CHILD_WORDS = ['my baby', 'my child', 'my son', 'my daughter', 'the baby', 'my small girl', 'my small boy']
PREG_WORDS = ['I am', 'I have been', 'me I am']

def T(intent, subject, texts):
    return [(x, intent, subject) for x in texts]

data = []
for s in CHILD_SYMPTOMS:
    for c in random.sample(CHILD_WORDS, 3):
        data += T('start_health_check', 'child', [
            f'{c} has {s}', f'{c} is {s} since yesterday', f'{c} {s} what should I do',
            f'please help {c} has {s} since morning'])
for s in PREG_SYMPTOMS:
    data += T('start_health_check', 'pregnancy', [
        f'I have {s}', f'{s} since last night what do I do', f'I am pregnant and I have {s}',
        f'please help me I have {s}'])
data += T('start_health_check', 'unknown', [
    'I am not feeling well', 'somebody is sick in my house', 'I feel sick', 'we are not well today',
    'I think something is wrong', 'I need to check my health', 'run a health check', 'start the checker'])
data += T('open_add_child', 'child', [
    'add my child', 'register my baby', 'I want to add my new baby', 'I just delivered last week',
    'I gave birth yesterday', 'my baby was born on monday', 'put my daughter in the app',
    'new baby in the house', 'I delivered a baby boy', 'add another child', 'register my son',
    'how do I add my baby', 'I want to track my child'])
data += T('open_add_pregnancy', 'pregnancy', [
    'I am pregnant', 'I think I am pregnant', 'track my pregnancy', 'I missed my period two months now',
    'start pregnancy tracking', 'I am expecting', 'me I am carrying a baby', 'add my pregnancy',
    'I am three months pregnant', 'I want to follow my pregnancy', 'new pregnancy'])
data += T('plan_diet', 'unknown', [
    'what should I cook today', 'plan my meals', 'what can we eat', 'food for my baby',
    'I have small money what can I cook', 'help me with food', 'what do I feed her',
    'meal plan please', 'what soup is good', 'we only have maize and beans at home',
    'my money is not enough for fish', 'give me one day meal plan', 'what should my child eat today',
    'plan food for the week', 'what should a pregnant woman eat'])
data += T('read_today', 'unknown', [
    'what is happening today', 'do I have a visit today', 'when is my next clinic day',
    'read my reminders', 'what do I have this week', 'any appointment coming',
    'when do I go to the clinic again', 'tell me my schedule', 'weighing day is when',
    'when is the next weighing'])
data += T('get_tip', 'unknown', [
    'give me a tip', 'feeding advice please', 'any advice for today', 'teach me something',
    'what should I know today', 'tips for feeding my baby', 'advice for breastfeeding',
    'how do I keep my baby healthy'])
data += T('log_weight', 'child', [
    'save my baby weight', 'record the weight from weighing', 'my child weighs 8 kilos now',
    'they weighed her today 6.5', 'log weight', 'update his weight', 'the nurse said 7.2 kg today'])
data += T('set_reminder', 'unknown', [
    'remind me to go to the clinic', 'set a reminder for friday', 'dont let me forget my appointment',
    'remind me tomorrow morning', 'alarm me for the weighing day', 'put a reminder for next tuesday'])
data += T('greeting', 'unknown', [
    'hello', 'hi nana', 'good morning', 'good evening nana', 'how are you', 'thank you',
    'thank you nana', 'God bless you', 'you have helped me', 'ok', 'yes please', 'goodnight'])
data += T('help_other', 'unknown', [
    'what can you do', 'help', 'how does this app work', 'I need help with the app',
    'what is this', 'teach me how to use this', 'where do I see my children',
    'can I use this without internet', 'change my language', 'who made this app'])
print('template examples:', len(data))

## 2. Optional: distill phrasing diversity from the online Nana LLM
Uses the SAME NVIDIA endpoint/model as the backend. Leave the key empty to skip.

In [ ]:
NVIDIA_API_KEY = ''  # <-- backend .env NVIDIA_API_KEY (optional)
NVIDIA_MODEL = 'meta/llama-3.3-70b-instruct'  # match backend env.nvidia.model

if NVIDIA_API_KEY:
    import requests
    gen_prompt = (
        'You create training data for a maternal-health app used by mothers in Tamale, Northern Ghana. '
        'Generate 40 short, realistic things a caregiver might type or say, in simple Ghanaian English '
        '(occasional pidgin flavour, small typos ok). For each, label intent and subject.\n'
        f'Allowed intents: {INTENTS}\nAllowed subjects: {SUBJECTS}\n'
        'Vary phrasing away from textbook English. Output ONLY JSON lines, one per example: '
        '{"text": ..., "intent": ..., "subject": ...}')
    added = 0
    for batch in range(8):
        r = requests.post('https://integrate.api.nvidia.com/v1/chat/completions',
            headers={'Authorization': f'Bearer {NVIDIA_API_KEY}'},
            json={'model': NVIDIA_MODEL, 'temperature': 1.0,
                  'messages': [{'role': 'user', 'content': gen_prompt + f'\nBatch {batch}.'}]},
            timeout=120)
        for line in r.json()['choices'][0]['message']['content'].splitlines():
            line = line.strip().strip('`')
            if not line.startswith('{'): continue
            try:
                ex = json.loads(line)
                if ex.get('intent') in INTENTS and ex.get('subject') in SUBJECTS and ex.get('text'):
                    data.append((ex['text'], ex['intent'], ex['subject'])); added += 1
            except Exception: pass
    print('LLM-distilled examples added:', added)
else:
    print('No API key — training on templates only.')

## 3. Train the multi-head model (~800 KB quantized)

In [ ]:
random.shuffle(data)
X = np.stack([featurize(t) for t, _, _ in data])
yi = np.array([INTENTS.index(i) for _, i, _ in data])
ys = np.array([SUBJECTS.index(s) for _, _, s in data])
Xtr, Xte, yitr, yite, ystr, yste = train_test_split(X, yi, ys, test_size=0.15,
                                                    stratify=yi, random_state=42)

inp = tf.keras.Input(shape=(BUCKETS,))
h = tf.keras.layers.Dense(96, activation='relu')(inp)
h = tf.keras.layers.Dropout(0.2)(h)
intent_out = tf.keras.layers.Dense(len(INTENTS), activation='softmax', name='intent')(h)
subject_out = tf.keras.layers.Dense(len(SUBJECTS), activation='softmax', name='subject')(h)
combined = tf.keras.layers.Concatenate(name='combined')([intent_out, subject_out])
model = tf.keras.Model(inp, combined)

def split_loss(y_true, y_pred):
    yi_t, ys_t = y_true[:, 0], y_true[:, 1]
    pi, ps = y_pred[:, :len(INTENTS)], y_pred[:, len(INTENTS):]
    li = tf.keras.losses.sparse_categorical_crossentropy(yi_t, pi)
    ls = tf.keras.losses.sparse_categorical_crossentropy(ys_t, ps)
    return li + 0.4 * ls

model.compile(optimizer=tf.keras.optimizers.Adam(2e-3), loss=split_loss)
y_tr = np.stack([yitr, ystr], axis=1).astype('float32')
y_te = np.stack([yite, yste], axis=1).astype('float32')
model.fit(Xtr, y_tr, validation_data=(Xte, y_te), epochs=40, batch_size=64, verbose=0,
          callbacks=[tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])

pred = model.predict(Xte, verbose=0)
print('=== INTENT ===')
print(classification_report(yite, pred[:, :len(INTENTS)].argmax(1), target_names=INTENTS, digits=3))
print('=== SUBJECT ===')
print(classification_report(yste, pred[:, len(INTENTS):].argmax(1), target_names=SUBJECTS, digits=3))

**Ship check:** intent accuracy should be ≥0.9 overall, and `start_health_check`
recall ≥0.95 (missing a symptom sentence is the bad error — though the app keeps
its keyword safety net in front of this model regardless).

In [ ]:
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
tfl = conv.convert()
open('nana_nlu.tflite', 'wb').write(tfl)
sha = hashlib.sha256(tfl).hexdigest()
manifest = {
    'name': 'nana-nlu', 'version': 1, 'kind': 'tflite',
    'sizeBytes': len(tfl), 'sha256': sha,
    'intents': INTENTS, 'subjects': SUBJECTS, 'buckets': BUCKETS,
    'featurizer': 'lowercase; [^a-z0-9\\x27 ]->space; unigrams u:, bigrams b:_, char trigrams c: of ^tok$; fnv1a32 % buckets; L2 norm',
    'outputLayout': 'concat: intents then subjects',
    'minConfidence': 0.5,
    'trainedOn': f'{len(data)} examples (templates + LLM distillation)',
    'disclaimer': 'Understanding only — replies are curated in-app; model never generates text.',
}
json.dump(manifest, open('manifest.json', 'w'), indent=2)
print(f'nana_nlu.tflite: {len(tfl)/1024:.0f} KB\nsha256: {sha}')

# Sanity: TFLite output must match Keras
it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
i_d, o_d = it.get_input_details()[0], it.get_output_details()[0]
for text in ['my baby is hot and not eating', 'what should I cook with small money',
             'when is my clinic day', 'I just delivered a boy']:
    it.set_tensor(i_d['index'], featurize(text).reshape(1, -1)); it.invoke()
    out = it.get_tensor(o_d['index'])[0]
    print(f'{text!r:45s} -> {INTENTS[out[:len(INTENTS)].argmax()]:20s} '
          f'({out[:len(INTENTS)].max():.2f}) subj={SUBJECTS[out[len(INTENTS):].argmax()]}')

## 4. Upload + register (same flow as the risk model)
Cloudinary unsigned preset, then register via `backend/scripts/register_model.js`
(edit its MANIFEST to these values) or the admin endpoint below.

In [ ]:
import requests
CLOUD_NAME = ''     # <-- your Cloudinary cloud name
UPLOAD_PRESET = ''  # <-- unsigned upload preset
up = requests.post(
    f'https://api.cloudinary.com/v1_1/{CLOUD_NAME}/raw/upload',
    files={'file': ('nana_nlu_v1.tflite', tfl)},
    data={'upload_preset': UPLOAD_PRESET, 'public_id': 'growwithme/models/nana_nlu_v1'},
).json()
print(up.get('secure_url') or up)
manifest['url'] = up['secure_url']
json.dump(manifest, open('manifest.json', 'w'), indent=2)
print(json.dumps(manifest, indent=2))